# 10 - LangGraph y Flujos de Trabajo

## Curso de LLMs y Aplicaciones de IA

**Duración estimada:** 2.5-3 horas

---

## Índice

1. [Introducción a LangGraph](#intro)
2. [Estados y Grafos](#estados)
3. [Flujo RAG con LangGraph](#rag)
4. [Auto-corrección](#correccion)
5. [Checkpoints y Persistencia](#checkpoints)
6. [Ejercicios prácticos](#ejercicios)

---

## Objetivos de aprendizaje

Al finalizar este notebook, serás capaz de:
- Crear grafos de estados con LangGraph
- Implementar flujos condicionales
- Añadir auto-corrección a sistemas RAG
- Usar checkpoints para persistencia

<a name="intro"></a>
## 1. Introducción a LangGraph

**LangGraph** es una librería de LangChain para crear flujos de trabajo como grafos de estados.

### ¿Por qué LangGraph?

- **Control explícito**: Define exactamente el flujo
- **Condicionales**: Diferentes caminos según resultados
- **Ciclos**: Permite iteraciones y re-intentos
- **Estado**: Mantiene información entre nodos
- **Persistencia**: Checkpoints para recuperación

In [1]:
# Install
#!pip install -q langchain langchain-groq langgraph langchain-huggingface faiss-cpu

In [2]:
import os
from getpass import getpass
import warnings
warnings.filterwarnings('ignore')

if 'GROQ_API_KEY' not in os.environ:
    os.environ['GROQ_API_KEY'] = getpass("GROQ API Key: ")

from langchain_groq import ChatGroq
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)
print("Configurado ✓")

GROQ API Key:  ········


Configurado ✓


<a name="estados"></a>
## 2. Estados y Grafos

En LangGraph, definimos:
- **State**: Datos que fluyen por el grafo
- **Nodes**: Funciones que procesan el estado
- **Edges**: Conexiones entre nodos

In [3]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

# Define state
class SimpleState(TypedDict):
    messages: List[str]
    current_step: str

# Define nodes
def step_one(state: SimpleState) -> SimpleState:
    messages = state["messages"] + ["Paso 1 completado"]
    return {"messages": messages, "current_step": "one"}

def step_two(state: SimpleState) -> SimpleState:
    messages = state["messages"] + ["Paso 2 completado"]
    return {"messages": messages, "current_step": "two"}

def step_three(state: SimpleState) -> SimpleState:
    messages = state["messages"] + ["Paso 3 completado"]
    return {"messages": messages, "current_step": "three"}

# Build graph
workflow = StateGraph(SimpleState)
workflow.add_node("step_one", step_one)
workflow.add_node("step_two", step_two)
workflow.add_node("step_three", step_three)

# Add edges
workflow.add_edge(START, "step_one")
workflow.add_edge("step_one", "step_two")
workflow.add_edge("step_two", "step_three")
workflow.add_edge("step_three", END)

# Compile
app = workflow.compile()
print("Grafo compilado ✓")

Grafo compilado ✓


In [4]:
# Run the graph
result = app.invoke({"messages": ["Inicio"], "current_step": ""})

print("Resultado:")
for msg in result["messages"]:
    print(f"  - {msg}")

Resultado:
  - Inicio
  - Paso 1 completado
  - Paso 2 completado
  - Paso 3 completado


### Grafos con condicionales

In [5]:
from typing import Literal

class ConditionalState(TypedDict):
    query: str
    query_type: str
    response: str

def classify_query(state: ConditionalState) -> ConditionalState:
    """Classify the query type."""
    query = state["query"].lower()
    if "precio" in query or "costo" in query:
        return {**state, "query_type": "pricing"}
    elif "horario" in query or "hora" in query:
        return {**state, "query_type": "schedule"}
    else:
        return {**state, "query_type": "general"}

def handle_pricing(state: ConditionalState) -> ConditionalState:
    return {**state, "response": "Los precios son: Básico 99€, Pro 299€, Enterprise consultar."}

def handle_schedule(state: ConditionalState) -> ConditionalState:
    return {**state, "response": "Horario: Lunes a Viernes, 9:00 a 18:00."}

def handle_general(state: ConditionalState) -> ConditionalState:
    return {**state, "response": "Para más información, contacta con soporte@empresa.com"}

def route_query(state: ConditionalState) -> Literal["pricing", "schedule", "general"]:
    return state["query_type"]

# Build conditional graph
cond_workflow = StateGraph(ConditionalState)
cond_workflow.add_node("classify", classify_query)
cond_workflow.add_node("pricing", handle_pricing)
cond_workflow.add_node("schedule", handle_schedule)
cond_workflow.add_node("general", handle_general)

cond_workflow.add_edge(START, "classify")
cond_workflow.add_conditional_edges(
    "classify",
    route_query,
    {"pricing": "pricing", "schedule": "schedule", "general": "general"}
)
cond_workflow.add_edge("pricing", END)
cond_workflow.add_edge("schedule", END)
cond_workflow.add_edge("general", END)

cond_app = cond_workflow.compile()
print("Grafo condicional compilado ✓")

Grafo condicional compilado ✓


In [6]:
# Test conditional routing
queries = [
    "¿Cuál es el precio del plan básico?",
    "¿Cuál es el horario de atención?",
    "¿Tienen servicio en México?"
]

for q in queries:
    result = cond_app.invoke({"query": q, "query_type": "", "response": ""})
    print(f"Q: {q}")
    print(f"A: {result['response']}\n")

Q: ¿Cuál es el precio del plan básico?
A: Los precios son: Básico 99€, Pro 299€, Enterprise consultar.

Q: ¿Cuál es el horario de atención?
A: Horario: Lunes a Viernes, 9:00 a 18:00.

Q: ¿Tienen servicio en México?
A: Para más información, contacta con soporte@empresa.com



<a name="rag"></a>
## 3. Flujo RAG con LangGraph

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage

# Create vector store
docs = [
    Document(page_content="El IBI se paga anualmente basado en el valor catastral."),
    Document(page_content="El IVTM grava la titularidad de vehículos matriculados."),
    Document(page_content="El ICIO se liquida al finalizar construcciones u obras."),
    Document(page_content="Las bonificaciones pueden reducir hasta un 90% el impuesto."),
]

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("Vector store creado ✓")

2026-06-24 19:37:08.654569: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Vector store creado ✓


In [8]:
from typing import List
from langchain_core.messages import BaseMessage

class RAGState(TypedDict):
    messages: List[BaseMessage]
    context: str
    response: str

def retrieve_context(state: RAGState) -> RAGState:
    """Retrieve relevant documents."""
    query = state["messages"][-1].content
    docs = retriever.invoke(query)
    context = "\n".join([d.page_content for d in docs])
    return {**state, "context": context}

def generate_response(state: RAGState) -> RAGState:
    """Generate response using LLM."""
    query = state["messages"][-1].content
    context = state["context"]
    
    prompt = f"""Responde basándote en el contexto.
    
Contexto: {context}

Pregunta: {query}

Respuesta:"""
    
    response = llm.invoke(prompt)
    return {**state, "response": response.content}

# Build RAG graph
rag_workflow = StateGraph(RAGState)
rag_workflow.add_node("retrieve", retrieve_context)
rag_workflow.add_node("generate", generate_response)

rag_workflow.add_edge(START, "retrieve")
rag_workflow.add_edge("retrieve", "generate")
rag_workflow.add_edge("generate", END)

rag_app = rag_workflow.compile()
print("RAG graph compilado ✓")

RAG graph compilado ✓


In [9]:
# Test RAG
result = rag_app.invoke({
    "messages": [HumanMessage(content="¿Qué es el IBI?")],
    "context": "",
    "response": ""
})

print(f"Respuesta: {result['response']}")

Respuesta: El IBI (Impuesto sobre Bienes Inmuebles) es un impuesto que se paga anualmente y está basado en el valor catastral de un inmueble.


<a name="correccion"></a>
## 4. Auto-corrección

Añadimos un paso de verificación y corrección.

In [10]:
class CorrectionState(TypedDict):
    query: str
    context: str
    response: str
    corrected_response: str
    needs_correction: bool

def retrieve(state: CorrectionState) -> CorrectionState:
    docs = retriever.invoke(state["query"])
    context = "\n".join([d.page_content for d in docs])
    return {**state, "context": context}

def generate(state: CorrectionState) -> CorrectionState:
    prompt = f"Contexto: {state['context']}\nPregunta: {state['query']}\nRespuesta:"
    response = llm.invoke(prompt)
    return {**state, "response": response.content}

def check_response(state: CorrectionState) -> CorrectionState:
    """Check if response needs correction."""
    check_prompt = f"""¿La siguiente respuesta está basada en el contexto?
    
Contexto: {state['context']}
Respuesta: {state['response']}

Responde solo 'SI' o 'NO'."""
    
    check = llm.invoke(check_prompt)
    needs_correction = "NO" in check.content.upper()
    return {**state, "needs_correction": needs_correction}

def correct_response(state: CorrectionState) -> CorrectionState:
    """Correct the response."""
    correct_prompt = f"""Mejora esta respuesta basándote solo en el contexto.
    
Contexto: {state['context']}
Respuesta original: {state['response']}

Respuesta mejorada:"""
    
    corrected = llm.invoke(correct_prompt)
    return {**state, "corrected_response": corrected.content}

def route_correction(state: CorrectionState) -> Literal["correct", "end"]:
    return "correct" if state["needs_correction"] else "end"

# Build correction graph
corr_workflow = StateGraph(CorrectionState)
corr_workflow.add_node("retrieve", retrieve)
corr_workflow.add_node("generate", generate)
corr_workflow.add_node("check", check_response)
corr_workflow.add_node("correct", correct_response)

corr_workflow.add_edge(START, "retrieve")
corr_workflow.add_edge("retrieve", "generate")
corr_workflow.add_edge("generate", "check")
corr_workflow.add_conditional_edges("check", route_correction, {"correct": "correct", "end": END})
corr_workflow.add_edge("correct", END)

corr_app = corr_workflow.compile()
print("Grafo con corrección compilado ✓")

Grafo con corrección compilado ✓


In [11]:
# Test
result = corr_app.invoke({
    "query": "¿Cuándo se paga el IVTM?",
    "context": "",
    "response": "",
    "corrected_response": "",
    "needs_correction": False
})

print(f"Respuesta original: {result['response']}")
print(f"Necesitó corrección: {result['needs_correction']}")
if result['corrected_response']:
    print(f"Respuesta corregida: {result['corrected_response']}")

Respuesta original: El IVTM se paga anualmente, al igual que el IBI, pero se basa en la titularidad de vehículos matriculados en lugar del valor catastral de una propiedad. Por lo general, el plazo para el pago del IVTM varía según la comunidad autónoma o la región en la que se encuentra el vehículo, pero suele ser anual.
Necesitó corrección: False


<a name="checkpoints"></a>
## 5. Checkpoints y Persistencia

In [12]:
from langgraph.checkpoint.memory import MemorySaver

# Create checkpointer
memory = MemorySaver()

# Compile with checkpointer
rag_with_memory = rag_workflow.compile(checkpointer=memory)

# Run with thread_id for session tracking
config = {"configurable": {"thread_id": "session1"}}

result = rag_with_memory.invoke({
    "messages": [HumanMessage(content="¿Qué impuestos hay?")],
    "context": "",
    "response": ""
}, config=config)

print(f"Respuesta: {result['response']}")

Respuesta: Hay impuestos como el IBI (Impuesto sobre Bienes Inmuebles), que se paga anualmente basado en el valor catastral, y otros impuestos que pueden ser reducidos mediante bonificaciones, que pueden alcanzar hasta un 90% de reducción.


<a name="ejercicios"></a>
## 6. Ejercicios Prácticos

### Ejercicio: Crear un flujo con múltiples pasos

In [13]:
# Exercise: Create a workflow that:
# 1. Receives a question
# 2. Classifies the question type
# 3. Retrieves relevant info
# 4. Generates response
# 5. Checks quality
# 6. Corrects if needed

In [14]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

# Catálogo de productos de SegurPlus
docs_segurplus = [
    # Seguros de Coche
    Document(page_content="SegurPlus Coche Básico: Incluye responsabilidad civil obligatoria, asistencia en carretera desde el kilómetro 0 y defensa jurídica."),
    Document(page_content="SegurPlus Coche Todo Riesgo: Incluye cobertura de daños propios, coche de sustitución Premium, rotura de lunas, robo e incendio sin franquicia."),
    
    # Seguros de Hogar
    Document(page_content="SegurPlus Hogar Esencial: Cubre daños por agua, incendio, inundaciones de instalaciones básicas y responsabilidad civil del inmueble hasta 150.000€."),
    Document(page_content="SegurPlus Hogar Confort: Incluye todo lo del plan Esencial más cobertura de robo dentro y fuera del hogar, servicio de manitas 24/7 y rotura de vitrocerámica y cristales.")
]

# Creamos el nuevo vector store de la aseguradora
vectorstore = FAISS.from_documents(docs_segurplus, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 1}) # Traemos el producto más relevante

In [15]:
from typing import TypedDict, Literal, List
from langgraph.graph import StateGraph, START, END

# Definición del Estado
class SegurPlusState(TypedDict):
    query: str
    query_type: str        # 'seguros' (coche/hogar) o 'general' (saludos, horarios, etc.)
    context: str           # Información de la póliza recuperada
    response: str          # Respuesta inicial de la IA
    corrected_response: str# Respuesta en caso de pasar por control de calidad
    needs_correction: bool # Flag de control de calidad

# ==========================================
# NODOS DEL GRAFO
# ==========================================

def classify_segurplus(state: SegurPlusState) -> SegurPlusState:
    """Deja que el LLM decida inteligentemente si es una duda de pólizas o una pregunta general/externa."""
    
    prompt = f"""Analiza la siguiente consulta de un cliente. 
    Determina si el usuario está preguntando por contratar, consultar coberturas o informarse sobre un SEGURO de coche o de hogar.
    Si pregunta por consejos de cocina, opiniones, saludos o temas fuera de los seguros, clasifícalo como GENERAL.

    Consulta: "{state['query']}"
    
    Responde estrictamente con una sola palabra: 'seguros' o 'general'."""
    
    response = llm.invoke(prompt)
    query_type = response.content.strip().lower()
    
    # Nos aseguramos de limpiar la respuesta por si el LLM añade puntos o mayúsculas
    if "seguros" in query_type:
        return {**state, "query_type": "seguros"}
    else:
        return {**state, "query_type": "general"}


def retrieve_policy(state: SegurPlusState) -> SegurPlusState:
    """Busca en el catálogo de FAISS la póliza que mejor responda al cliente."""
    docs = retriever.invoke(state["query"])
    context = "\n".join([d.page_content for d in docs])
    return {**state, "context": context}


def generate_segurplus_resp(state: SegurPlusState) -> SegurPlusState:
    """Genera la respuesta como un agente de atención al cliente de SegurPlus."""
    if state["query_type"] == "seguros":
        prompt = f"""Eres el asistente virtual de la aseguradora SegurPlus. 
        Responde la duda del cliente basándote estrictamente en el catálogo de coberturas provisto.
        
        Catálogo SegurPlus: {state['context']}
        Pregunta del cliente: {state['query']}
        Respuesta comercial detallada:"""
    else:
        prompt = f"""Eres el asistente virtual de SegurPlus. Responde de manera amable, corporativa y servicial 
        a la siguiente consulta general (puedes mencionar que abrimos de L-V de 9:00 a 18:00 si preguntan por contacto): {state['query']}"""
    
    response = llm.invoke(prompt)
    return {**state, "response": response.content}


def check_segurplus_quality(state: SegurPlusState) -> SegurPlusState:
    """Valida si el asistente inventó coberturas o si se mantuvo fiel al catálogo."""
    if state["query_type"] != "seguros":
        return {**state, "needs_correction": False}
        
    check_prompt = f"""Analiza si la respuesta del asistente virtual contiene información falsa o inventada que NO esté en el Catálogo de SegurPlus.
    
    Catálogo Oficial: {state['context']}
    Respuesta dada: {state['response']}
    
    ¿La respuesta inventa o exagera coberturas? Responde estrictamente 'SI' o 'NO'."""
    
    check = llm.invoke(check_prompt)
    needs_correction = "SI" in check.content.upper()
    return {**state, "needs_correction": needs_correction}


def correct_segurplus_resp(state: SegurPlusState) -> SegurPlusState:
    """Reescribe la respuesta asegurando la precisión del catálogo de SegurPlus."""
    correct_prompt = f"""Corrige la respuesta para que se ciña ÚNICAMENTE a lo estipulado en el Catálogo Oficial de SegurPlus. Sé transparente.
    
    Catálogo Oficial: {state['context']}
    Respuesta errónea: {state['response']}
    
    Respuesta corregida y segura:"""
    
    corrected = llm.invoke(correct_prompt)
    return {**state, "corrected_response": corrected.content}

# ==========================================
# ROUTERS
# ==========================================
def route_by_insurance_type(state: SegurPlusState) -> Literal["retrieve", "generate"]:
    return "retrieve" if state["query_type"] == "seguros" else "generate"

def route_by_insurance_quality(state: SegurPlusState) -> Literal["correct", "end"]:
    return "correct" if state["needs_correction"] else "end"

# ==========================================
# CONSTRUCCIÓN DEL GRAFO
# ==========================================
segurplus_workflow = StateGraph(SegurPlusState)

segurplus_workflow.add_node("classify", classify_segurplus)
segurplus_workflow.add_node("retrieve", retrieve_policy)
segurplus_workflow.add_node("generate", generate_segurplus_resp)
segurplus_workflow.add_node("check", check_segurplus_quality)
segurplus_workflow.add_node("correct", correct_segurplus_resp)

segurplus_workflow.add_edge(START, "classify")
segurplus_workflow.add_conditional_edges("classify", route_by_insurance_type, {"retrieve": "retrieve", "generate": "generate"})
segurplus_workflow.add_edge("retrieve", "generate")
segurplus_workflow.add_edge("generate", "check")
segurplus_workflow.add_conditional_edges("check", route_by_insurance_quality, {"correct": "correct", "end": END})
segurplus_workflow.add_edge("correct", END)

segurplus_app = segurplus_workflow.compile()
print("¡Grafo de SegurPlus compilado y listo para producción! ✓")

¡Grafo de SegurPlus compilado y listo para producción! ✓


In [16]:
# --- Caso 1: Pregunta sobre Seguro de Coche ---
print("=== CLIENTE 1 ===")
res_coche = segurplus_app.invoke({
    "query": "¿El plan a todo riesgo me cubre el coche si me lo roban?", 
    "query_type": "", "context": "", "response": "", "corrected_response": "", "needs_correction": False
})
print(f"Ruta tomada: {res_coche['query_type'].upper()}")
print(f"Respuesta de SegurPlus: {res_coche['response']}\n")

# --- Caso 2: Pregunta sobre Seguro de Hogar ---
print("=== CLIENTE 2 ===")
res_hogar = segurplus_app.invoke({
    "query": "Necesito un seguro de hogar que incluya servicio de manitas, ¿tienen alguno?", 
    "query_type": "", "context": "", "response": "", "corrected_response": "", "needs_correction": False
})
print(f"Ruta tomada: {res_hogar['query_type'].upper()}")
print(f"Respuesta de SegurPlus: {res_hogar['response']}\n")

# --- Caso 3: Charla casual / General ---
print("=== CLIENTE 3 ===")
res_general = segurplus_app.invoke({
    "query": "Hola, ¿a qué hora cierran sus oficinas hoy?", 
    "query_type": "", "context": "", "response": "", "corrected_response": "", "needs_correction": False
})

# --- Caso 4: Limites y rerespuesta 
print("=== CLIENTE 4 ===")
res_general = segurplus_app.invoke({
    "query": "¿Es seguro para mi vitroceramica cocinar a fuego lento durante muchas horas?", 
    "query_type": "", "context": "", "response": "", "corrected_response": "", "needs_correction": False
})
print(f"Ruta tomada: {res_general['query_type'].upper()}")
print(f"Respuesta de SegurPlus: {res_general['response']}")

=== CLIENTE 1 ===
Ruta tomada: SEGUROS
Respuesta de SegurPlus: Estimado cliente,

Gracias por considerar a SegurPlus para proteger su vehículo. Me alegra informarle que nuestro plan SegurPlus Coche Todo Riesgo ofrece una amplia gama de coberturas para darle tranquilidad en la carretera.

En cuanto a su pregunta, sí, el plan a todo riesgo incluye cobertura en caso de robo de su vehículo. Esto significa que, en el evento de que su coche sea robado, SegurPlus estará allí para apoyarlo y ayudarlo a recuperar su pérdida.

Además de la cobertura contra robo, nuestro plan a todo riesgo también incluye:

- Cobertura de daños propios: para proteger su vehículo en caso de accidentes o daños.
- Coche de sustitución Premium: para que pueda seguir moviéndose con comodidad mientras su vehículo está en reparación.
- Rotura de lunas: para cubrir el costo de reparar o reemplazar las lunas de su vehículo en caso de daño.
- Incendio sin franquicia: para proteger su vehículo en caso de incendio, sin neces

## Resumen

En este notebook hemos aprendido:

1. **LangGraph**: Crear flujos como grafos de estados
2. **Condicionales**: Routing basado en resultados
3. **RAG workflow**: Retrieve → Generate
4. **Auto-corrección**: Verificar y mejorar respuestas
5. **Checkpoints**: Persistencia de sesiones

En el siguiente notebook veremos **RAG Avanzado Agentic** con flujos completos.

---

## Referencias

- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)

In [17]:
import session_info
session_info.show(html=False)

-----
ipykernel                   7.2.0
langchain_community         0.4.1
langchain_core              1.4.0
langchain_groq              1.1.2
langchain_huggingface       NA
langgraph                   NA
session_info                v1.0.1
-----
IPython             9.1.0
jupyter_client      8.8.0
jupyter_core        5.9.1
jupyterlab          4.5.6
notebook            7.5.5
-----
Python 3.12.9 | packaged by Anaconda, Inc. | (main, Feb  6 2025, 13:04:33) [Clang 14.0.6 ]
macOS-10.16-x86_64-i386-64bit
-----
Session information updated at 2026-06-24 19:37
